#Chat Inspections

##Data Collection

In [11]:
!pip install textblob

import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.feature_extraction.text import TfidfVectorizer
from textblob import TextBlob
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import (
  accuracy_score,
  precision_score,
  recall_score,
  f1_score,
  roc_auc_score,
  classification_report
)

df = pd.read_parquet('https://huggingface.co/datasets/ddds-Capstone/Datasets/resolve/main/chat_metrics.parquet')


##TF-IDF
Term Frequency-Inverse Document Frequency

In [2]:
#Question TF-IDF
tfidf_question = TfidfVectorizer(max_features=500, stop_words='english')

question_tfidf = tfidf_question.fit_transform(df['question_en'].fillna(''))

question_features = tfidf_question.get_feature_names_out()

#Weight of each word across all questions
scores = np.asarray(question_tfidf.sum(axis=0)).flatten()

top_words = pd.DataFrame({'word': question_features, 'tfidf_score':scores}).sort_values('tfidf_score', ascending=False)

#Answer TF-IDF
tfidf_answer = TfidfVectorizer(max_features=500, stop_words='english')

answer_tfidf = tfidf_answer.fit_transform(df['answer_en'].fillna(''))

answer_features = tfidf_answer.get_feature_names_out()

#Weight of each word across all questions
scores = np.asarray(answer_tfidf.sum(axis=0)).flatten()

top_words = pd.DataFrame({'word': answer_features, 'tfidf_score':scores}).sort_values('tfidf_score', ascending=False)
top_words.head(20)

,word,tfidf_score
449,td,38.171470
264,information,27.945269
445,sunport,27.544662
71,albuquerque,24.184012
471,tsa,22.620104
58,abq,22.298030
453,terminal,21.776694
268,international,20.188348
256,https,19.798388
140,com,19.229386


##Sentiment Analysis

In [3]:
df['polarity'] = df['answer_en'].apply(lambda x: TextBlob(str(x)).sentiment.polarity)
df['subjectivity'] = df['answer_en'].apply(lambda x: TextBlob(str(x)).sentiment.subjectivity)

df['polarity'] = df['question_en'].apply(lambda x: TextBlob(str(x)).sentiment.polarity)
df['subjectivity'] = df['question_en'].apply(lambda x: TextBlob(str(x)).sentiment.subjectivity)

##Make a Copy

In [4]:
df_clean = df.copy()

##Data Cleaning

In [5]:
#Keep sessions with star_rating only
df_clean.dropna(subset=['star_rating'], inplace=True)

#Binary Satisfaction
df_clean['satisfaction'] = (df_clean['star_rating'] >= 4).astype(int)

df_clean['satisfaction'].value_counts(normalize=True)

#Set target
target = 'satisfaction'

##Processing

In [6]:
features = [

  #Time
  'day_sin',
  'day_cos',
  'hour_sin',
  'hour_cos',
  'month_sin',
  'month_cos',

  #Performance
  'processing_time_seconds',
  'total_tokens',
  'input_tokens',
  'output_tokens',
  'model_calls',
  'tool_calls_count',

  #Q&A
  'question_length',
  'answer_length',

  #Maps
  'has_geolocation',

  #Primary Category
  'primary_category_greetings',
  'primary_category_sunport_amenities',
  'primary_category_navigation',
  'primary_category_airline_logistics',
  'primary_category_general_info',

  #Selected Agent
  'selected_agentreporter',
  'selected_agentplanner',
  'selected_agentlocation',
  'selected_agentlocation_fallback_to_reporter',
  'selected_agentbroad_search_synthesis',
  'selected_agentbroad_search_passthrough',

  #Sentiment
  'polarity',
  'subjectivity'
]

X = df_clean[features]
y = df_clean['satisfaction']

###Train/test split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
  X,
  y,
  test_size=0.20,
  random_state=42,
  stratify=y #Maintains roughly same proportion
)

###Standard Scaler

In [10]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

###Neural Network

In [14]:
nn = Sequential([
  Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
  Dropout(0.2),
  Dense(32, activation='relu'),
  Dense(1, activation='sigmoid')
])

#Compile
nn.compile(
  optimizer='adam',
  loss='binary_crossentropy',
  metrics=['accuracy']
)

#Train
early_stop = EarlyStopping(
  monitor='val_loss',
  patience=10,
  restore_best_weights=True
)

#Fit
history = nn.fit(
  X_train_scaled,
  y_train,
  validation_split=0.2,
  epochs=100,
  batch_size=32,
  callbacks=[early_stop],
  verbose=1
)

#Probability
y_prob_nn = nn.predict(X_test_scaled).ravel()

#Predict
y_pred_nn = (y_prob_nn >= 0.5).astype(int)

#Evaluate
print('Accuracy:', accuracy_score(y_test, y_pred_nn))
print('Precision:', precision_score(y_test, y_pred_nn))
print('Recall:', recall_score(y_test, y_pred_nn))
print('F1:', f1_score(y_test, y_pred_nn))
print('ROC-AUC:', roc_auc_score(y_test, y_prob_nn))

print('\nClassification Report:')
print(classification_report(y_test, y_pred_nn))

Epoch 1/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 59ms/step - accuracy: 0.4819 - loss: 0.7139 - val_accuracy: 0.5072 - val_loss: 0.6930
Epoch 2/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.5906 - loss: 0.6591 - val_accuracy: 0.5507 - val_loss: 0.6659
Epoch 3/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6558 - loss: 0.6313 - val_accuracy: 0.6087 - val_loss: 0.6484
Epoch 4/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.6667 - loss: 0.6211 - val_accuracy: 0.6377 - val_loss: 0.6337
Epoch 5/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.6812 - loss: 0.5961 - val_accuracy: 0.6667 - val_loss: 0.6215
Epoch 6/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.7138 - loss: 0.5698 - val_accuracy: 0.6957 - val_loss: 0.6116
Epoch 7/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.7319 - loss: 0.5635 - val_accuracy: 0.6957 - val_loss: 0.6023
Epoch 8/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.7283 - loss: 0.5478 - val_accuracy: 0.6957 - val_loss: